In [18]:
%load_ext autoreload
%autoreload 2
import pandas as pd

archetype_df = pd.read_excel("../data/archetypes_v1.xlsx", index_col=0)
# vorbereitete thermische Endenergie (Heizen + DHW)
archetype_df["E_therm_MWhyr"] = (
    archetype_df["E_final_heating_MWhyr"] + archetype_df["E_final_dhw_MWhyr"]
)  # [file:2]
#maybe save this for future use?

archetype_df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,usage_type_tabula,usage_type_statistik,usage_simplified,building_period,building_status,energy_carrier_primary,Af_m2,Aroof_m2,GFA_m2,Aocc_m2,people0,E_final_heating_MWhyr,E_final_dhw_MWhyr,E_final_cooling_MWhyr,E_final_user_MWhyr,Q_use_heating_MWhyr,Q_use_dhw_MWhyr,Q_use_cooling_MWhyr,Q_use_user_MWhyr,E_therm_MWhyr
archetype_id,,,,,,,,,,,,,,,,,,,,
EFH I_Bis 1919_fossil,Einfamilienhaus,Wohngebäude mit einer Wohnung,residential,bis 1919,original,fossil,210.08,103.9,366.5,208.0,2.97,78.612111,8.535414,0.0,9.002447,62.889689,6.828332,0.0,9.002447,87.147525
EFH II_1920-44_fossil,Einfamilienhaus,Wohngebäude mit einer Wohnung,residential,1920-44,original,fossil,210.08,103.9,366.5,208.0,2.97,64.099791,8.492056,0.0,9.002447,51.279833,6.793644,0.0,9.002447,72.591846
EFH III_1945-59_fossil,Einfamilienhaus,Wohngebäude mit einer Wohnung,residential,1945-59,original,fossil,210.08,103.9,366.5,208.0,2.97,58.556169,8.498683,0.0,9.002447,46.844935,6.798946,0.0,9.002447,67.054852
EFH IV_1960-79_fossil,Einfamilienhaus,Wohngebäude mit einer Wohnung,residential,1960-79,original,fossil,210.08,103.9,366.5,208.0,2.97,47.740755,8.467251,0.0,9.002447,38.192604,6.773800,0.0,9.002447,56.208006
EFH V_1980-89_fossil,Einfamilienhaus,Wohngebäude mit einer Wohnung,residential,1980-89,original,fossil,210.08,103.9,366.5,208.0,2.97,53.573437,8.481319,0.0,9.002447,42.858750,6.785055,0.0,9.002447,62.054756


In [19]:
from archifer.parser import parse_yaml_config
targets, weights, constraints = parse_yaml_config("constraints.yaml")
constraints

[{'name': 'Af_total',
  'type': 'sum',
  'column': 'Af_m2',
  'hard': False,
  'target': 'Af_total'},
 {'name': 'share_residential_Af',
  'type': 'share',
  'column': 'Af_m2',
  'predicate': "usage_simplified == 'residential'",
  'hard': False,
  'target': 'share_residential_Af'},
 {'name': 'share_energy_therm',
  'type': 'share_group',
  'column': 'E_therm_MWhyr',
  'categories': {'fossil': "energy_carrier_primary == 'fossil'",
   'biogene': "energy_carrier_primary == 'biogene'",
   'district_heat': "energy_carrier_primary == 'district_heat'",
   'heat_pump': "energy_carrier_primary == 'heat_pump'"},
  'hard': False,
  'targets': 'share_energy_therm'}]

In [20]:
from archifer.parser import create_model

model = create_model(targets=targets, 
                     weights=weights, 
                     constraints=constraints, 
                     archetype_df=archetype_df)
model


Archetype_Inference:
MINIMIZE
10*s_Af_total_neg + 10*s_Af_total_pos + 100*s_share_energy_therm_biogene_neg + 100*s_share_energy_therm_biogene_pos + 100*s_share_energy_therm_district_heat_neg + 100*s_share_energy_therm_district_heat_pos + 100*s_share_energy_therm_fossil_neg + 100*s_share_energy_therm_fossil_pos + 100*s_share_energy_therm_heat_pump_neg + 100*s_share_energy_therm_heat_pump_pos + 10*s_share_residential_Af_neg + 10*s_share_residential_Af_pos + 0.0
SUBJECT TO
Af_total: s_Af_total_neg - s_Af_total_pos + 210.08 w_EFH_III_1945_59_biogene
 + 210.08 w_EFH_III_1945_59_dh + 210.08 w_EFH_III_1945_59_fossil
 + 210.08 w_EFH_III_1945_59_hp + 210.08 w_EFH_III_1945_59_reno_biogene
 + 210.08 w_EFH_III_1945_59_reno_dh + 210.08 w_EFH_III_1945_59_reno_fossil
 + 210.08 w_EFH_III_1945_59_reno_hp + 210.08 w_EFH_II_1920_44_biogene
 + 210.08 w_EFH_II_1920_44_dh + 210.08 w_EFH_II_1920_44_fossil
 + 210.08 w_EFH_II_1920_44_hp + 210.08 w_EFH_II_1920_44_reno_biogene
 + 210.08 w_EFH_II_1920_44_reno_dh 

In [21]:

import pulp as pl

solver = pl.PULP_CBC_CMD() # timeLimit=1) # Zeitlimit in Sekunden; best-effort Lösung; wenn exakter gewünscht, Zeitlimit entfernen
model.solve(solver)

print("Status:", pl.LpStatus[model.status])

Status: Optimal


In [22]:
w = pl.LpVariable.dicts("w", archetype_df.index.to_list(), lowBound=0, cat="Integer") # hier cat = "Continuous" für LP-Relaxation

w_sol = pd.Series({i: w[i].varValue for i in archetype_df.index.to_list()})
w_sol

EFH I_Bis 1919_fossil      None
EFH II_1920-44_fossil      None
EFH III_1945-59_fossil     None
EFH IV_1960-79_fossil      None
EFH V_1980-89_fossil       None
                           ... 
MWB II_1920-44_reno_hp     None
MWB III_1945-59_reno_hp    None
MWB IV_1960-79_reno_hp     None
MWB V_1980-89_reno_hp      None
MWB VI_1990-99_reno_hp     None
Length: 228, dtype: object